# DL Layer Training: CNN-BiGRU

In [1]:
from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/kwago/'

Mounted at /content/drive


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
!pip install tensorflow keras-tuner -q
!pip install tf2onnx onnx -q
import tf2onnx
import onnx
import numpy as np
import pandas as pd
import json, warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
import keras_tuner as kt
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, classification_report, confusion_matrix
)

print(f'TensorFlow: {tf.__version__}')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.4/129.4 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 839.1/839.1 kB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 93.6 MB/s eta 0:00:00
TensorFlow: 2.20.0


## Load Preprocessed Data

In [4]:
X_train_B = np.load(BASE + 'X_train_B.npy')
X_val_B   = np.load(BASE + 'X_val_B.npy')
X_test_B  = np.load(BASE + 'X_test_B.npy')
y_train_B = np.load(BASE + 'y_train_B.npy')
y_val     = np.load(BASE + 'y_val.npy')
y_test    = np.load(BASE + 'y_test.npy')

print(f'Train : {X_train_B.shape}  label dist: {dict(zip(*np.unique(y_train_B, return_counts=True)))}')
print(f'Val   : {X_val_B.shape}   label dist: {dict(zip(*np.unique(y_val, return_counts=True)))}')
print(f'Test  : {X_test_B.shape}  label dist: {dict(zip(*np.unique(y_test, return_counts=True)))}')

MAX_WORDS = 10000
MAX_LEN   = 150

Train : (7634, 150)  label dist: {np.int64(0): np.int64(6207), np.int64(1): np.int64(1427)}
Val   : (1636, 150)   label dist: {np.int64(0): np.int64(1330), np.int64(1): np.int64(306)}
Test  : (1637, 150)  label dist: {np.int64(0): np.int64(1331), np.int64(1): np.int64(306)}


## Class Weights

Since SMOTE was removed from Pipeline B, we balance training via class weights passed into `model.fit()`.

In [5]:
weights = compute_class_weight('balanced', classes=np.array([0, 1]), y=y_train_B)
class_weight_dict = {0: float(weights[0]), 1: float(weights[1])}
print(f'Class weights: {class_weight_dict}')

Class weights: {0: 0.614950861930079, 1: 2.674842326559215}


## Custom F1 Metric

Keras Tuner needs a Keras-tracked metric as its objective. We define F1 as a custom metric so Hyperband can optimize on it directly.

In [14]:
@tf.keras.utils.register_keras_serializable()
class F1Score(tf.keras.metrics.Metric):
    def __init__(self, name='f1_score', **kwargs):
        super().__init__(name=name, **kwargs)
        self.precision_m = tf.keras.metrics.Precision()
        self.recall_m    = tf.keras.metrics.Recall()

    def update_state(self, y_true, y_pred, sample_weight=None):
        self.precision_m.update_state(y_true, y_pred, sample_weight)
        self.recall_m.update_state(y_true, y_pred, sample_weight)

    def result(self):
        p = self.precision_m.result()
        r = self.recall_m.result()
        return tf.math.divide_no_nan(2 * p * r, p + r)

    def reset_state(self):
        self.precision_m.reset_state()
        self.recall_m.reset_state()

## CNN-BiGRU Model Builder (for Keras Tuner)

**Architecture** (manuscript §2.5):
`Embedding → Conv1D → MaxPool1D → Bidirectional GRU → Dropout → Dense(64) → Dropout → Dense(1, sigmoid)`

**Tunable hyperparameters:**
- `embed_dim` — embedding size
- `filters` — Conv1D filters
- `kernel_size` — Conv1D kernel size
- `gru_units` — BiGRU hidden units
- `dropout` — dropout rate
- `learning_rate` — Adam LR

In [7]:
def build_model(hp):
    embed_dim   = hp.Choice('embed_dim',    [64, 128, 256])
    filters     = hp.Choice('filters',      [64, 128, 256])
    kernel_size = hp.Choice('kernel_size',  [3, 5])
    gru_units   = hp.Choice('gru_units',    [64, 128, 256])
    dropout     = hp.Float('dropout',       0.2, 0.5, step=0.1)
    lr          = hp.Choice('learning_rate',[1e-4, 5e-4, 1e-3])

    model = models.Sequential([
        layers.Embedding(MAX_WORDS, embed_dim, input_length=MAX_LEN),
        layers.Conv1D(filters, kernel_size, activation='relu', padding='same'),
        layers.MaxPooling1D(pool_size=2),
        layers.Bidirectional(layers.GRU(gru_units)),
        layers.Dropout(dropout),
        layers.Dense(64, activation='relu'),
        layers.Dropout(dropout),
        layers.Dense(1, activation='sigmoid'),
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss='binary_crossentropy',
        metrics=[
            'accuracy',
            tf.keras.metrics.Precision(name='precision'),
            tf.keras.metrics.Recall(name='recall'),
            F1Score(),
        ],
    )
    return model

## Keras Tuner — Hyperband Search

Hyperband is an efficient early-stopping variant of random search. It runs many configurations briefly, then extends the most promising ones.

In [8]:
tuner = kt.Hyperband(
    build_model,
    objective=kt.Objective('val_f1_score', direction='max'),
    max_epochs=20,
    factor=3,
    directory=BASE + 'kt_dir',
    project_name='cnn_bigru',
    overwrite=True,
)

tuner.search(
    X_train_B, y_train_B,
    validation_data=(X_val_B, y_val),
    class_weight=class_weight_dict,
    callbacks=[
        callbacks.EarlyStopping(monitor='val_f1_score', patience=3, mode='max'),
    ],
    verbose=1,
)

best_hp = tuner.get_best_hyperparameters(1)[0]
print('\nBest hyperparameters:')
for k, v in best_hp.values.items():
    print(f'  {k:<18}: {v}')

Trial 30 Complete [00h 00m 26s]
val_f1_score: 0.9273020625114441

Best val_f1_score So Far: 0.9499192237854004
Total elapsed time: 00h 13m 43s

Best hyperparameters:
  embed_dim         : 64
  filters           : 128
  kernel_size       : 5
  gru_units         : 64
  dropout           : 0.4
  learning_rate     : 0.0005
  tuner/epochs      : 3
  tuner/initial_epoch: 0
  tuner/bracket     : 2
  tuner/round       : 0


## Train Best Model to Completion

Rebuild with best hyperparameters and train with `EarlyStopping`, `ReduceLROnPlateau`, and `ModelCheckpoint`.

In [9]:
best_model = tuner.hypermodel.build(best_hp)
best_model.summary()

history = best_model.fit(
    X_train_B, y_train_B,
    validation_data=(X_val_B, y_val),
    epochs=50,
    batch_size=64,
    class_weight=class_weight_dict,
    callbacks=[
        callbacks.EarlyStopping(
            monitor='val_f1_score', patience=5, mode='max',
            restore_best_weights=True
        ),
        callbacks.ReduceLROnPlateau(
            monitor='val_f1_score', factor=0.5, patience=3,
            mode='max', min_lr=1e-6, verbose=1
        ),
        callbacks.ModelCheckpoint(
            BASE + 'cnn_bigru_best.keras',
            monitor='val_f1_score', mode='max',
            save_best_only=True, verbose=1
        ),
    ],
    verbose=1,
)

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
117/120 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.7372 - f1_score: 0.3652 - loss: 0.6219 - precision: 0.3221 - recall: 0.4279
Epoch 1: val_f1_score improved from None to 0.90513, saving model to /content/drive/MyDrive/kwago/cnn_bigru_best.keras

Epoch 1: finished saving model to /content/drive/MyDrive/kwago/cnn_bigru_best.keras
120/120 ━━━━━━━━━━━━━━━━━━━━ 7s 32ms/step - accuracy: 0.8173 - f1_score: 0.5845 - loss: 0.4607 - precision: 0.5083 - recall: 0.6875 - val_accuracy: 0.9627 - val_f1_score: 0.9051 - val_loss: 0.1195 - val_precision: 0.8635 - val_recall: 0.9510 - learning_rate: 5.0000e-04
Epoch 2/50
118/120 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.9680 - f1_score: 0.9183 - loss: 0.1239 - precision: 0.8887 - recall: 0.9501
Epoch 2: val_f1_score improved from 0.90513 to 0.93671, saving model to /content/drive/MyDrive/kwago/cnn_bigru_best.keras

Epoch 2: finished saving model to /content/drive/MyDrive/kwago/cnn_bigru_best.keras
120/120 ━━━━━━━━━━━━━━━━━━━━ 2s 1

## Final Evaluation — Test Set

In [10]:
y_test_prob = best_model.predict(X_test_B, verbose=0).flatten()
y_test_pred = (y_test_prob >= 0.5).astype(int)

metrics_final = {
    'accuracy' : accuracy_score(y_test, y_test_pred),
    'precision': precision_score(y_test, y_test_pred, zero_division=0),
    'recall'   : recall_score(y_test, y_test_pred, zero_division=0),
    'f1'       : f1_score(y_test, y_test_pred, zero_division=0),
    'auc_roc'  : roc_auc_score(y_test, y_test_prob),
}

print('=== CNN-BiGRU — TEST SET ===')
for k, v in metrics_final.items():
    print(f'  {k:<12}: {v:.4f}')
print(f'\nConfusion Matrix:\n{confusion_matrix(y_test, y_test_pred)}')
print(f'\n{classification_report(y_test, y_test_pred, target_names=["Benign","Phishing"], zero_division=0)}')

=== CNN-BiGRU — TEST SET ===
  accuracy    : 0.9737
  precision   : 0.9021
  recall      : 0.9641
  f1          : 0.9321
  auc_roc     : 0.9952

Confusion Matrix:
[[1299   32]
 [  11  295]]

              precision    recall  f1-score   support

      Benign       0.99      0.98      0.98      1331
    Phishing       0.90      0.96      0.93       306

    accuracy                           0.97      1637
   macro avg       0.95      0.97      0.96      1637
weighted avg       0.97      0.97      0.97      1637



## Save Model and Artifacts

In [11]:
# Full model (Keras native format — recommended over .h5 for TF2)
best_model.save(BASE + 'cnn_bigru_model.keras')
print('Saved: cnn_bigru_model.keras')

# Best hyperparameters (for reproducibility and documentation)
with open(BASE + 'cnn_bigru_best_hp.json', 'w') as f:
    json.dump(best_hp.values, f, indent=2)
print('Saved: cnn_bigru_best_hp.json')

best_model.export(BASE + 'cnn_bigru_savedmodel')

!python -m tf2onnx.convert \
    --saved-model "{BASE}cnn_bigru_savedmodel" \
    --output "{BASE}cnn_bigru_model.onnx" \
    --opset 13

# Results CSV
results = pd.DataFrame([{
    'Model'    : 'CNN-BiGRU',
    'Set'      : 'Test',
    **{k: round(v, 4) for k, v in metrics_final.items()},
}])
results.to_csv(BASE + 'cnn_bigru_results.csv', index=False)
print('\nResults summary:')
print(results.to_string(index=False))

Saved: cnn_bigru_model.keras
Saved: cnn_bigru_best_hp.json
Saved artifact at '/content/drive/MyDrive/kwago/cnn_bigru_savedmodel'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 150), dtype=tf.float32, name='keras_tensor_9')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  139680455757072: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139680535657552: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139680535654288: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139680455769552: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139680455756880: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139680455764560: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139680455766288: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139680455763792: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139680455765136: TensorSpec(shape=(), dtype=tf.resource, name=None)


# Changing to CPU formatting

In [19]:
import tensorflow as tf
import tf2onnx
import os

model_path = os.path.join(BASE, 'cnn_bigru_best.keras')
onnx_output_path = os.path.join(BASE, 'cnn_bigru_model.onnx')

# 1. Disable GPU to force standard RNN kernels
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'
tf.keras.backend.clear_session()

with tf.device('/cpu:0'):
    # 2. Rebuild the exact architecture (avoiding the saved CuDNN graph)
    # We use the best hyperparameters found earlier
    model = tf.keras.Sequential([
        tf.keras.layers.Embedding(MAX_WORDS, 64, input_length=MAX_LEN),
        tf.keras.layers.Conv1D(128, 5, activation='relu', padding='same'),
        tf.keras.layers.MaxPooling1D(pool_size=2),
        tf.keras.layers.Bidirectional(tf.keras.layers.GRU(64)),
        tf.keras.layers.Dropout(0.4),
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.Dropout(0.4),
        tf.keras.layers.Dense(1, activation='sigmoid'),
    ])

    # 3. Load weights from the saved model instead of loading the whole model object
    if os.path.exists(model_path):
        model.load_weights(model_path)

        # 4. Convert using the fresh 'clean' graph
        spec = (tf.TensorSpec((None, 150), tf.float32, name="input"),)

        onnx_model, _ = tf2onnx.convert.from_keras(
            model,
            input_signature=spec,
            opset=13
        )

        with open(onnx_output_path, 'wb') as f:
            f.write(onnx_model.SerializeToString())

        print(f'Successfully converted to ONNX (using standard GRU kernels): {onnx_output_path}')
    else:
        print(f'Error: File not found at {model_path}')

ERROR:tf2onnx.tfonnx:Tensorflow op [sequential_1_1/bidirectional_1_1/forward_gru_1_1/CudnnRNNV3: CudnnRNNV3] is not supported
ERROR:tf2onnx.tfonnx:Tensorflow op [sequential_1_1/bidirectional_1_1/backward_gru_1_1/CudnnRNNV3: CudnnRNNV3] is not supported
ERROR:tf2onnx.tfonnx:Unsupported ops: Counter({'CudnnRNNV3': 2})


Successfully converted and saved to /content/drive/MyDrive/kwago/cnn_bigru_model.onnx


In [29]:
import os
import tensorflow as tf
import tf2onnx

# 1. Disable GPU and reset session
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'
tf.keras.backend.clear_session()

model_path = os.path.join(BASE, 'cnn_bigru_best.keras')
onnx_path = os.path.join(BASE, 'cnn_bigru_model.onnx')

with tf.device('/cpu:0'):
    # 2. Rebuild architecture using standard GRU settings to avoid CuDNN
    # CuDNN requires: activation='tanh', recurrent_activation='sigmoid',
    # recurrent_dropout=0, unroll=False, and reset_after=True.
    # By rebuilding on CPU, we ensure the compatible kernel is used.
    model = tf.keras.Sequential([
        tf.keras.layers.Embedding(MAX_WORDS, 64, input_shape=(MAX_LEN,)),
        tf.keras.layers.Conv1D(128, 5, activation='relu', padding='same'),
        tf.keras.layers.MaxPooling1D(pool_size=2),
        tf.keras.layers.Bidirectional(tf.keras.layers.GRU(
            64,
            activation='tanh',
            recurrent_activation='sigmoid',
            reset_after=True,
            unroll=True  # <--- ADD THIS LINE!
        )),
        tf.keras.layers.Dropout(0.4),
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.Dropout(0.4),
        tf.keras.layers.Dense(1, activation='sigmoid'),
    ])

    if os.path.exists(model_path):
        # 3. Load weights into the CPU-safe architecture
        model.load_weights(model_path)
        print("Weights successfully loaded into CPU-compatible architecture.")

        # 4. Convert directly to ONNX using a concrete function
        # This is the most reliable path to avoid Keras 3 naming issues
        @tf.function
        def serve(input_tensor):
            return model(input_tensor, training=False)

        spec = (tf.TensorSpec((None, MAX_LEN), tf.float32, name="input"),)

        onnx_model, _ = tf2onnx.convert.from_function(
            serve,
            input_signature=spec,
            opset=13,
            output_path=onnx_path
        )

        print(f"Success! Clean ONNX model created at: {onnx_path}")
    else:
        print(f"Error: Weights file not found at {model_path}")

Weights successfully loaded into CPU-compatible architecture.
Success! Clean ONNX model created at: /content/drive/MyDrive/kwago/cnn_bigru_model.onnx
